# Занятие 3. Лабораторная: линейная классификация

Лабораторная домашняя. В ней девятнадцать задач, все обязательные, бонусов нет.
Задачи короткие и сгруппированы в шесть блоков: функции руками, градиентный
спуск на данных, регрессия вместо классификации, порог, регуляризация,
три сорта.

Как и в прошлый раз, часть задач — реализация, часть — эксперимент с выбором
по метрике. Проверка пересчитывает каждый эксперимент независимо.

**Заготовки.** В каждой заготовке указано, что должно получиться: тип
и форма в аннотации, размеры входов и выходов в комментарии перед функцией.
Как считать — в теории перед задачей. Имена переменных и функций менять
нельзя, проверка ищет их по именам. После каждой задачи идет ячейка
с открытыми проверками; при сдаче работа дополнительно проверяется закрытыми
тестами на других данных.

Основные данные — диагностика опухолей молочной железы из sklearn: 569
наблюдений, 30 признаков, посчитанных по снимку, и ответ: злокачественная
опухоль или нет. Мы кодируем **злокачественную единицей**: искать нужно именно
ее. Вино вернется в последнем блоке.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
SEED = 42

In [ ]:
cancer = load_breast_cancer()
cancer_names = [str(name) for name in cancer.feature_names]
y_cancer = (cancer.target == 0).astype(int)       # 1 — злокачественная

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    cancer.data, y_cancer, test_size=0.3, random_state=SEED, stratify=y_cancer
)
scaler = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# столбец единиц для свободного члена, как на семинаре
A_train = np.column_stack([np.ones(len(X_train)), X_train])
A_test = np.column_stack([np.ones(len(X_test)), X_test])

print("train:", X_train.shape, " test:", X_test.shape)
print(f"доля злокачественных: {y_cancer.mean():.2f}")

Сигмоида и log loss — те же, что на семинаре, они понадобятся дальше.

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=float)
    with np.errstate(over="ignore"):
        return 1 / (1 + np.exp(-z))


def logistic_loss(y_true: np.ndarray, proba: np.ndarray, eps: float = 1e-12) -> float:
    y_true = np.asarray(y_true, dtype=float)
    proba = np.clip(np.asarray(proba, dtype=float), eps, 1 - eps)
    return float(-np.mean(y_true * np.log(proba) + (1 - y_true) * np.log(1 - proba)))

## Блок A. Функции руками

### Задача 1. Отступы

Отступ объекта $M_i = y_i^{\pm}\,(\langle w, x_i \rangle + b)$, где
$y^{\pm} = 2y - 1$ переводит метки $\{0, 1\}$ в $\{-1, +1\}$.

In [ ]:
# X: (n, d), y: (n,) из нулей и единиц, w: (d,), b — число
# возвращает отступы: (n,)
def margins(X: np.ndarray, y: np.ndarray, w: np.ndarray, b: float) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
w_check, b_check = np.array([1.0, -1.0]), 0.5
X_check = np.array([[1.0, 0.0], [0.0, 1.0], [2.0, 2.0]])
y_check = np.array([1, 0, 1])

# 1. счет руками: (1 + 0.5) * 1, (-1 + 0.5) * (-1), (0 + 0.5) * 1
assert np.allclose(margins(X_check, y_check, w_check, b_check), [1.5, 0.5, 0.5]), "отступы посчитаны неверно"

# 2. форма
assert margins(X_train, y_train, np.zeros(30), 0.0).shape == (len(y_train),), "отступов столько же, сколько объектов"

# 3. знак отступа — верно ли классифицирован объект
model_check = LogisticRegression(max_iter=5000).fit(X_train, y_train)
m_check = margins(X_train, y_train, model_check.coef_[0], model_check.intercept_[0])
assert np.array_equal(m_check > 0, model_check.predict(X_train) == y_train), "знак отступа должен совпадать с правильностью ответа"

print("проверки пройдены")

### Задача 2. Две функции потерь через отступ

Обе усредняются по объектам:

$$\text{logistic} = \frac{1}{n}\sum \log(1 + e^{-M_i}), \qquad
\text{squared} = \frac{1}{n}\sum (1 - M_i)^2$$

Для логистической потери при большом отрицательном отступе $e^{-M}$
переполняется. Устойчивый способ — `np.logaddexp(0, -M)`: это ровно
$\log(e^0 + e^{-M})$, посчитанный без переполнения.

In [ ]:
# M: (n,) отступы -> число
def logistic_margin_loss(M: np.ndarray) -> float:
    return ...


def squared_margin_loss(M: np.ndarray) -> float:
    return ...

In [ ]:
# --- проверка ---
M_check = np.array([2.0, 0.5, -1.0])

# 1. счет руками
assert np.isclose(squared_margin_loss(M_check), (1 + 0.25 + 4) / 3), "squared посчитан неверно"
assert np.isclose(logistic_margin_loss(M_check), np.mean(np.log1p(np.exp(-M_check)))), "logistic посчитан неверно"

# 2. логистическая потеря через отступ совпадает с log loss через вероятности
assert np.isclose(logistic_margin_loss(m_check), logistic_loss(y_train, model_check.predict_proba(X_train)[:, 1])), (
    "log(1 + exp(-M)) и log loss через вероятности — одно и то же"
)

# 3. большие по модулю отступы не ломают
assert np.isfinite(logistic_margin_loss(np.array([-1000.0, 1000.0]))), "переполнение: используйте np.logaddexp"

# 4. squared штрафует уверенно правильные ответы, logistic почти нет
assert squared_margin_loss(np.array([5.0])) > 1 and logistic_margin_loss(np.array([5.0])) < 0.01

print("проверки пройдены")

### Задача 3. Градиент log loss со штрафом

На семинаре градиент был $\frac{1}{n}X^\top(p - y)$, где $X$ — матрица
со столбцом единиц, а $w$ включал свободный член. В лабораторной, как и в прошлый
раз, матрицу со столбцом единиц обозначаем $A$, а полный вектор весов вместе
со свободным членом — $\theta$; $w$ здесь — все компоненты $\theta$, кроме
первой. Добавим штраф L2 на веса: к функции потерь прибавляется
$\frac{\alpha}{2}\|w\|^2$, свободный член не штрафуется. Градиент штрафа
по $w$ равен $\alpha w$, по свободному члену — ноль.

In [ ]:
# A: (n, d + 1) со столбцом единиц, y: (n,) из нулей и единиц, theta: (d + 1,), alpha — число
# возвращает градиент log loss плюс штраф: (d + 1,)
def logistic_gradient_l2(A: np.ndarray, y: np.ndarray, theta: np.ndarray, alpha: float) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
theta_check = np.linspace(-0.5, 0.5, A_train.shape[1])

# 1. при alpha = 0 это градиент семинара
plain = A_train.T @ (sigmoid(A_train @ theta_check) - y_train) / len(y_train)
assert np.allclose(logistic_gradient_l2(A_train, y_train, theta_check, 0.0), plain), "при alpha=0 должен получаться обычный градиент"

# 2. штраф добавляет alpha * theta, кроме свободного члена
diff = logistic_gradient_l2(A_train, y_train, theta_check, 0.3) - plain
assert np.isclose(diff[0], 0.0) and np.allclose(diff[1:], 0.3 * theta_check[1:]), "штраф должен действовать на все веса, кроме первого"

# 3. форма
assert logistic_gradient_l2(A_train, y_train, theta_check, 0.1).shape == theta_check.shape

print("проверки пройдены")

### Задача 4. Градиентный спуск

Спуск из нулевого вектора, `n_iter` шагов размера `lr` против градиента
из задачи 3. После каждого шага в историю дописывается log loss на обучающих
данных — только потери, без штрафа.

In [ ]:
# возвращает theta: (d + 1,) после n_iter шагов и history — список из n_iter значений log loss без штрафа
def fit_logistic_l2(A: np.ndarray, y: np.ndarray, lr: float, n_iter: int, alpha: float) -> tuple[np.ndarray, list[float]]:
    theta: np.ndarray = ...
    history: list[float] = ...
    for _ in range(n_iter):
        ...
    return theta, history

In [ ]:
# --- проверка ---
theta_check, history_check = fit_logistic_l2(A_train, y_train, lr=0.5, n_iter=300, alpha=0.01)

# 1. длина истории и форма весов
assert len(history_check) == 300 and theta_check.shape == (A_train.shape[1],), "history длины n_iter, theta длины d + 1"

# 2. первый шаг из нуля: theta = -lr * градиент в нуле
first_step = -0.5 * logistic_gradient_l2(A_train, y_train, np.zeros(A_train.shape[1]), 0.01)
theta_one, _ = fit_logistic_l2(A_train, y_train, lr=0.5, n_iter=1, alpha=0.01)
assert np.allclose(theta_one, first_step), "после одного шага theta = -lr * градиент в нуле"

# 3. потери убывают
assert history_check[-1] < history_check[0] and history_check[-1] < 0.15, "потери должны заметно упасть"

print("проверки пройдены")

### Задача 5. Предсказание

Вероятность класса 1 — сигмоида от $A\theta$. Ответ — единица там, где
вероятность не меньше порога.

In [ ]:
# A: (n, d + 1), theta: (d + 1,) -> вероятности класса 1: (n,)
def predict_proba_theta(A: np.ndarray, theta: np.ndarray) -> np.ndarray:
    return ...


# -> метки 0 и 1: (n,) целых чисел
def predict_theta(A: np.ndarray, theta: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
proba_check = predict_proba_theta(A_test, theta_check)
pred_check = predict_theta(A_test, theta_check)

# 1. вероятности в отрезке, метки целые
assert proba_check.shape == (len(y_test),) and (proba_check >= 0).all() and (proba_check <= 1).all()
assert set(np.unique(pred_check)) <= {0, 1} and pred_check.dtype.kind == "i", "метки должны быть целыми нулями и единицами"

# 2. порог работает
assert predict_theta(A_test, theta_check, threshold=0.0).all(), "при пороге 0 все ответы единицы"
assert not predict_theta(A_test, theta_check, threshold=1.01).any(), "при пороге выше единицы все ответы нули"

# 3. качество разумное
assert accuracy_score(y_test, pred_check) > 0.9, "после 300 шагов accuracy на тесте должна быть выше 0.9"

print("проверки пройдены")

## Блок B. Спуск на данных

### Задача 6. Без штрафа веса не останавливаются

Классы в этих данных почти разделимы. Без штрафа спуск может делать веса
все больше: каждое увеличение делает вероятности увереннее, а log loss
на обучении — меньше. Запустите `fit_logistic_l2` с `alpha=0` и `lr=0.5`
на `A_train` для трех длин спуска из `iteration_counts` и запишите норму
вектора весов без свободного члена, $\|\theta_{1:}\|$, в словарь
`weight_norms`: ключ — число итераций, значение — норма. В `weights_keep_growing`
запишите, растет ли норма с каждым удлинением спуска.

In [ ]:
iteration_counts = [100, 1000, 10000]

In [ ]:
...  # ваш код: запуски спуска для каждого числа итераций

weight_norms: dict[int, float] = ...        # ключ — число итераций из iteration_counts, значение — норма весов
weights_keep_growing: bool = ...

print({n: round(v, 2) for n, v in weight_norms.items()}, "растут:", weights_keep_growing)

In [ ]:
# --- проверка ---
assert set(weight_norms) == set(iteration_counts), "ключи — числа итераций из iteration_counts"
theta_1000, _ = fit_logistic_l2(A_train, y_train, lr=0.5, n_iter=1000, alpha=0.0)
assert np.isclose(weight_norms[1000], np.linalg.norm(theta_1000[1:])), "норма для 1000 итераций не совпала с пересчетом"
assert isinstance(weights_keep_growing, bool) and weights_keep_growing == (weight_norms[100] < weight_norms[1000] < weight_norms[10000])
print("проверки пройдены")

### Задача 7. Наш штраф и `C` в sklearn

В sklearn функция потерь логистической регрессии записана как
$C \sum_i \ell_i + \frac{1}{2}\|w\|^2$, у нас — $\frac{1}{n}\sum_i \ell_i + \frac{\alpha}{2}\|w\|^2$.
Поделив первую на $Cn$, получаем вторую с $\alpha = \frac{1}{Cn}$, то есть
$C = \frac{1}{\alpha n}$.

Обучите спуск с `alpha=0.01`, `lr=0.5`, `n_iter=5000` на `A_train` и сохраните
веса в `theta_l2`. Посчитайте `C_equivalent`, обучите
`LogisticRegression(C=C_equivalent, max_iter=100000, tol=1e-12)` на `X_train`
и сохраните модель в `sklearn_l2`. Ее веса в том же порядке, что у `theta`, —
свободный член, потом остальные, — положите в `theta_sklearn`. В `max_weight_diff` запишите наибольшее по модулю
расхождение между `theta_l2` и `theta_sklearn`.

In [ ]:
theta_l2: np.ndarray = ...                  # (d + 1,)
C_equivalent: float = ...
sklearn_l2: LogisticRegression = ...        # обученная модель sklearn с C_equivalent
theta_sklearn: np.ndarray = ...             # (d + 1,): свободный член, потом веса
max_weight_diff: float = ...

print(f"C = {C_equivalent:.4f}, наибольшее расхождение весов {max_weight_diff:.2e}")

In [ ]:
# --- проверка ---
assert np.isclose(C_equivalent, 1 / (0.01 * len(y_train))), "C_equivalent = 1 / (alpha * n)"
assert theta_l2.shape == theta_sklearn.shape == (A_train.shape[1],), "оба вектора длины d + 1"
assert np.isclose(max_weight_diff, np.abs(theta_l2 - theta_sklearn).max())
assert max_weight_diff < 1e-4, f"наш спуск и sklearn должны сойтись к одним весам, расхождение {max_weight_diff:.2e}"
print("проверки пройдены")

## Блок C. Регрессия вместо классификации

### Задача 8. Линейная регрессия на метках

Самый прямой способ классифицировать линейной моделью: обучить
`LinearRegression` на метках 0 и 1 и считать ответом единицу там, где
предсказание не меньше порога. Напишите две функции.

In [ ]:
# X: (n, d), y: (n,) из нулей и единиц -> обученная LinearRegression
def fit_regression_classifier(X: np.ndarray, y: np.ndarray) -> LinearRegression:
    return ...


# -> метки 0 и 1: (n,) целых чисел
def predict_regression_classifier(model: LinearRegression, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
regression_clf = fit_regression_classifier(X_train, y_train)
assert isinstance(regression_clf, LinearRegression) and hasattr(regression_clf, "coef_"), "нужна обученная LinearRegression"
labels_reg = predict_regression_classifier(regression_clf, X_test)
assert labels_reg.shape == (len(y_test),) and set(np.unique(labels_reg)) <= {0, 1} and labels_reg.dtype.kind == "i"
assert np.array_equal(labels_reg, (regression_clf.predict(X_test) >= 0.5).astype(int)), "порог применяется к предсказанию регрессии"
print("проверки пройдены")

### Задача 9. На реальных данных

Обучите на `X_train` две модели: регрессию на метках функцией из задачи 8,
сохраните в `regression_clf`, и `LogisticRegression()` с параметрами
по умолчанию, сохраните в `logistic_clf`. На тесте посчитайте accuracy каждой:
`acc_regression` и `acc_logistic`. В `out_of_range_share` положите долю тестовых
объектов, для которых регрессия предсказала число вне отрезка $[0, 1]$.

In [ ]:
regression_clf: LinearRegression = ...      # из fit_regression_classifier
logistic_clf: LogisticRegression = ...

acc_regression: float = ...
acc_logistic: float = ...
out_of_range_share: float = ...

print(f"accuracy: регрессия {acc_regression:.3f}, логистическая {acc_logistic:.3f}")
print(f"предсказаний регрессии вне [0, 1]: {out_of_range_share:.0%}")

In [ ]:
# --- проверка ---
assert np.isclose(acc_regression, accuracy_score(y_test, predict_regression_classifier(regression_clf, X_test)))
assert np.isclose(acc_logistic, accuracy_score(y_test, LogisticRegression().fit(X_train, y_train).predict(X_test)))
assert 0 < out_of_range_share < 1, "доля должна быть между нулем и единицей"
assert np.isclose(out_of_range_share, np.mean(np.abs(regression_clf.predict(X_test) - 0.5) > 0.5))
print("проверки пройдены")

На этих данных регрессия на метках не хуже логистической по accuracy, и это
не редкость: если классы почти разделимы, любая разумная прямая их делит.
Но треть предсказаний регрессии лежит вне отрезка $[0, 1]$, и это уже
не вероятности. Дальше два эксперимента, где разница видна.

### Задача 10. Далекие объекты

Данные ниже: два облака на плоскости, `X_near` и `y_near`, и те же облака
плюс тридцать объектов класса 1 далеко в углу, `X_with_far` и `y_with_far`.
Обучите на каждом наборе регрессию на метках и `LogisticRegression(C=1e4)`
и посчитайте accuracy на том же наборе, на котором обучали. Результат —
словарь `far_accuracy` с ключами `"regression_near"`, `"logistic_near"`,
`"regression_far"`, `"logistic_far"`. В `robust_choice` положите `"regression"`
или `"logistic"` — кто не потерял качество от далеких объектов.

In [ ]:
rng_far = np.random.default_rng(SEED)
cloud_0 = rng_far.normal([0, 0], 1, (60, 2))
cloud_1 = rng_far.normal([3, 3], 1, (60, 2))
cloud_far = rng_far.normal([14, 14], 1, (30, 2))

X_near = np.vstack([cloud_0, cloud_1])
y_near = np.r_[np.zeros(60, dtype=int), np.ones(60, dtype=int)]
X_with_far = np.vstack([cloud_0, cloud_1, cloud_far])
y_with_far = np.r_[y_near, np.ones(30, dtype=int)]

In [ ]:
...  # ваш код: четыре модели, по две на каждый набор

far_accuracy: dict[str, float] = ...    # ключи regression_near, logistic_near, regression_far, logistic_far
robust_choice: str = ...                # "regression" или "logistic"

print({k: round(v, 3) for k, v in far_accuracy.items()}, "устойчивее:", robust_choice)

In [ ]:
# --- проверка ---
assert set(far_accuracy) == {"regression_near", "logistic_near", "regression_far", "logistic_far"}, "четыре ключа"
assert robust_choice in ("regression", "logistic")
reg_far = fit_regression_classifier(X_with_far, y_with_far)
assert np.isclose(far_accuracy["regression_far"], accuracy_score(y_with_far, predict_regression_classifier(reg_far, X_with_far)))
assert far_accuracy["regression_far"] < far_accuracy["regression_near"], "далекие объекты должны испортить регрессию"
assert robust_choice == ("logistic" if far_accuracy["logistic_far"] >= far_accuracy["regression_far"] else "regression")
print("проверки пройдены")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (title, X_set, y_set) in zip(axes, [("два облака", X_near, y_near), ("плюс далекие объекты", X_with_far, y_with_far)]):
    reg = fit_regression_classifier(X_set, y_set)
    log = LogisticRegression(C=1e4).fit(X_set, y_set)
    ax.scatter(X_set[y_set == 0, 0], X_set[y_set == 0, 1], s=18, color=GREY, label="класс 0")
    ax.scatter(X_set[y_set == 1, 0], X_set[y_set == 1, 1], s=18, color=BLUE, label="класс 1")
    xs = np.linspace(-3, 7, 50)
    ax.plot(xs, -(reg.coef_[0] * xs + reg.intercept_ - 0.5) / reg.coef_[1], color=OCHRE, lw=2, label="регрессия на метках")
    ax.plot(xs, -(log.coef_[0, 0] * xs + log.intercept_[0]) / log.coef_[0, 1], color=BLACK, lw=2, label="логистическая")
    ax.set_xlim(-3.5, 7)
    ax.set_ylim(-3.5, 7)
    ax.set_title(f"{title}, границы в области облаков")
axes[0].legend()
plt.tight_layout()
plt.show()

### Задача 11. Каждая модель минимизирует свою потерю

Регрессия на метках $\pm 1$ минимизирует квадратичную потерю через отступ,
логистическая регрессия — логистическую. Обучите на `X_train`
`LinearRegression` на метках `2 * y_train - 1` и сохраните в `regression_pm`;
логистическую берем готовую, `logistic_clf` из задачи 9. Посчитайте на тесте
отступы каждой модели: `margins_regression` — это $y^{\pm}\cdot$ предсказание
регрессии, `margins_logistic` — $y^{\pm}\cdot$ `decision_function`
логистической. Сложите обе потери из задачи 2 в таблицу `loss_table`: строки
`"regression"` и `"logistic"`, столбцы `"squared"`, `"logistic"`.
В `own_loss_wins` запишите, верно ли, что у каждой модели ее собственная
потеря меньше, чем у другой.

In [ ]:
regression_pm: LinearRegression = ...       # обучена на метках -1 и +1
margins_regression: np.ndarray = ...        # (n_test,)
margins_logistic: np.ndarray = ...          # (n_test,)

loss_table: pd.DataFrame = ...              # (2, 2), строки regression и logistic, столбцы squared и logistic
own_loss_wins: bool = ...

print(loss_table.round(3), "\nкаждая выигрывает в своей потере:", own_loss_wins)

In [ ]:
# --- проверка ---
assert list(loss_table.index) == ["regression", "logistic"] and list(loss_table.columns) == ["squared", "logistic"]
check_reg = LinearRegression().fit(X_train, 2 * y_train - 1)
check_m = (2 * y_test - 1) * check_reg.predict(X_test)
assert np.isclose(loss_table.loc["regression", "squared"], squared_margin_loss(check_m)), "squared регрессии не совпал с пересчетом"
assert np.isclose(loss_table.loc["logistic", "logistic"], logistic_margin_loss((2 * y_test - 1) * logistic_clf.decision_function(X_test)))
assert isinstance(own_loss_wins, bool)
print("проверки пройдены")

### Задача 12. Вероятности от регрессии

Если предсказание регрессии на метках 0 и 1 считать вероятностью класса 1,
что получится? Прижмите предсказания `regression_clf` на тесте к отрезку
$[10^{-6}, 1 - 10^{-6}]$ и посчитайте `logloss_regression` через `logistic_loss`.
Рядом `logloss_logistic` — log loss вероятностей `logistic_clf`. В
`calibrated_choice` положите `"regression"` или `"logistic"`, у кого log loss
меньше.

In [ ]:
proba_from_regression: np.ndarray = ...     # (n_test,) предсказания regression_clf, прижатые к отрезку
logloss_regression: float = ...
logloss_logistic: float = ...
calibrated_choice: str = ...                # "regression" или "logistic"

print(f"log loss: регрессия {logloss_regression:.3f}, логистическая {logloss_logistic:.3f}, честнее {calibrated_choice}")

In [ ]:
# --- проверка ---
assert np.isclose(logloss_regression, logistic_loss(y_test, np.clip(regression_clf.predict(X_test), 1e-6, 1 - 1e-6)))
assert np.isclose(logloss_logistic, log_loss(y_test, logistic_clf.predict_proba(X_test)[:, 1]))
assert calibrated_choice == ("regression" if logloss_regression < logloss_logistic else "logistic")
print("проверки пройдены")

**Вопрос.** По accuracy в задаче 9 регрессия не уступала, а по log loss
в задаче 12 проигрывает в разы. Что именно log loss видит, чего не видит
accuracy? Объясните на примере одного объекта с предсказанием регрессии 1.4.

*Ответ пишите здесь.*

## Блок D. Порог

Порог не часть модели: его выбирают под задачу. Подбирать его на тестовой
выборке нельзя, поэтому обучающая часть делится еще раз: на `X_fit`
обучаем, на `X_val` выбираем порог, тест остается нетронутым.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)
threshold_model = LogisticRegression(max_iter=5000).fit(X_fit, y_fit)
proba_val = threshold_model.predict_proba(X_val)[:, 1]
proba_test = threshold_model.predict_proba(X_test)[:, 1]

threshold_grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
print("fit:", len(y_fit), " val:", len(y_val), " test:", len(y_test))

### Задача 13. Precision и recall при пороге

Без `sklearn.metrics`. Ответ единица там, где вероятность не меньше порога.
Precision — доля настоящих единиц среди предсказанных единиц, recall — доля
найденных среди всех настоящих единиц. Если единиц не предсказано ни одной,
precision считайте нулем.

In [ ]:
# y_true: (n,) из нулей и единиц, proba: (n,) вероятности класса 1, threshold — число
# возвращает (precision, recall) — два числа
def precision_recall_at(y_true: np.ndarray, proba: np.ndarray, threshold: float) -> tuple[float, float]:
    return ...

In [ ]:
# --- проверка ---
p_check, r_check = precision_recall_at(y_val, proba_val, 0.5)
pred_half = (proba_val >= 0.5).astype(int)
assert np.isclose(p_check, precision_score(y_val, pred_half)) and np.isclose(r_check, recall_score(y_val, pred_half)), "не совпало со sklearn"
assert precision_recall_at(y_val, proba_val, 1.01) == (0.0, 0.0), "если ничего не предсказано, precision и recall нули"
assert precision_recall_at(y_val, proba_val, 0.0)[1] == 1.0, "при пороге 0 recall равен единице"
print("проверки пройдены")

### Задача 14. Порог по F1

Для каждого порога из `threshold_grid` посчитайте на валидации precision,
recall и $F_1 = \frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$
(ноль, если оба нули). Сложите в `threshold_table`: индекс — порог, столбцы
`"precision"`, `"recall"`, `"f1"`. В `best_threshold_f1` положите порог
с наибольшим F1; при равенстве — меньший.

In [ ]:
...  # ваш код: precision, recall и F1 для каждого порога

threshold_table: pd.DataFrame = ...   # индекс threshold_grid, столбцы precision, recall, f1
best_threshold_f1: float = ...

print(threshold_table.round(3).T)
print("порог по F1:", best_threshold_f1)

In [ ]:
# --- проверка ---
assert list(threshold_table.columns) == ["precision", "recall", "f1"] and np.allclose(threshold_table.index, threshold_grid)
p3, r3 = precision_recall_at(y_val, proba_val, 0.3)
assert np.isclose(threshold_table.loc[0.3, "f1"], 2 * p3 * r3 / (p3 + r3)), "F1 для порога 0.3 не совпал"
assert np.isclose(best_threshold_f1, threshold_table["f1"].idxmax())
assert (threshold_table["recall"].diff().dropna() <= 1e-12).all(), "recall не может расти с ростом порога"
print("проверки пройдены")

### Задача 15. Порог по цене ошибки

В диагностике пропустить злокачественную опухоль дороже, чем зря отправить
на дообследование. Пусть пропуск, то есть ответ 0 при настоящей 1, стоит 10,
а ложная тревога — 1. Для каждого порога из `threshold_grid` посчитайте
суммарную цену ошибок на валидации и сложите в `cost_table`: `pd.Series`
с индексом-порогом. В `best_threshold_cost` положите порог с наименьшей ценой;
при равенстве — меньший.

In [ ]:
FN_COST, FP_COST = 10, 1

...  # ваш код: цена ошибок для каждого порога

cost_table: pd.Series = ...           # индекс threshold_grid, значения — суммарная цена
best_threshold_cost: float = ...

print(cost_table.to_dict())
print(f"порог по цене: {best_threshold_cost}, по F1 было {best_threshold_f1}")

In [ ]:
# --- проверка ---
assert np.allclose(cost_table.index, threshold_grid)
pred_half = (proba_val >= 0.5).astype(int)
expected_half = 10 * np.sum((pred_half == 0) & (y_val == 1)) + np.sum((pred_half == 1) & (y_val == 0))
assert cost_table.loc[0.5] == expected_half, "цена при пороге 0.5 не совпала с пересчетом"
assert np.isclose(best_threshold_cost, cost_table.idxmin())
assert best_threshold_cost < best_threshold_f1, "когда пропуск дорог, порог должен уйти ниже, чем по F1"
print("проверки пройдены")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(threshold_table.index, threshold_table["precision"], "o-", color=BLUE, ms=4, label="precision")
axes[0].plot(threshold_table.index, threshold_table["recall"], "o-", color=OCHRE, ms=4, label="recall")
axes[0].plot(threshold_table.index, threshold_table["f1"], "o-", color=BLACK, ms=4, label="F1")
axes[0].axvline(best_threshold_f1, color=BLACK, ls="--", lw=1)
axes[0].set_xlabel("порог")
axes[0].legend()
axes[1].plot(cost_table.index, cost_table.values, "o-", color=OCHRE, ms=4)
axes[1].axvline(best_threshold_cost, color=BLACK, ls="--", lw=1)
axes[1].set_xlabel("порог")
axes[1].set_ylabel("цена ошибок на валидации")
plt.tight_layout()
plt.show()

for name, threshold in [("по умолчанию", 0.5), ("по F1", best_threshold_f1), ("по цене", best_threshold_cost)]:
    precision, recall = precision_recall_at(y_test, proba_test, threshold)
    print(f"тест, порог {name:13s} {threshold:.2f}: precision {precision:.3f}, recall {recall:.3f}")

### Задача 16. Редкий класс

Сделаем злокачественные опухоли редкими: в обучении оставляем каждую десятую.
Индексы такой выборки готовы в `imbalanced_idx`. Обучите на
`X_train[imbalanced_idx]`, `y_train[imbalanced_idx]` две логистические регрессии
с `max_iter=5000`: обычную, `plain_model`, и с `class_weight="balanced"`,
`balanced_model`. Посчитайте на тесте recall злокачественных для каждой —
`recall_plain` и `recall_balanced` — и accuracy — `acc_plain` и `acc_balanced`.

In [ ]:
rng_imbalance = np.random.default_rng(SEED)
positive_idx = np.flatnonzero(y_train == 1)
kept_positive = rng_imbalance.choice(positive_idx, size=len(positive_idx) // 10, replace=False)
imbalanced_idx = np.r_[np.flatnonzero(y_train == 0), kept_positive]
print(f"в обучении осталось злокачественных: {len(kept_positive)} из {len(imbalanced_idx)}")

In [ ]:
plain_model: LogisticRegression = ...
balanced_model: LogisticRegression = ...

recall_plain: float = ...
recall_balanced: float = ...
acc_plain: float = ...
acc_balanced: float = ...

print(f"обычная:       recall {recall_plain:.3f}, accuracy {acc_plain:.3f}")
print(f"сбалансированная: recall {recall_balanced:.3f}, accuracy {acc_balanced:.3f}")

In [ ]:
# --- проверка ---
check_plain = LogisticRegression(max_iter=5000).fit(X_train[imbalanced_idx], y_train[imbalanced_idx])
assert np.isclose(recall_plain, recall_score(y_test, check_plain.predict(X_test))), "recall обычной модели не совпал с пересчетом"
assert np.isclose(acc_plain, accuracy_score(y_test, check_plain.predict(X_test)))
assert recall_balanced > recall_plain, "взвешивание классов должно поднять recall редкого класса"
print("проверки пройдены")

**Вопрос.** `class_weight="balanced"` умножает потери объектов редкого
класса на большой коэффициент. Как это связано с порогом из задачи 15: можно
ли получить похожий эффект, не переобучая модель?

*Ответ пишите здесь.*

## Блок E. Регуляризация

### Задача 17. Выбор `C` по кросс-валидации

Для каждого `C` из `C_grid` посчитайте средний log loss на кросс-валидации
с разбиением `cv_cancer` для `LogisticRegression(C=C, max_iter=10000)`
на `X_train`; sklearn возвращает `neg_log_loss` со знаком минус. Результат —
словарь `cv_logloss`, ключ `C`, значение log loss. В `best_C` положите `C`
с наименьшим значением.

In [ ]:
C_grid = [0.001, 0.01, 0.1, 1, 10, 100]
cv_cancer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [ ]:
...  # ваш код: кросс-валидация для каждого C

cv_logloss: dict[float, float] = ...    # ключ — C из C_grid, значение — средний log loss
best_C: float = ...

print({C: round(v, 3) for C, v in cv_logloss.items()}, "лучший C:", best_C)

In [ ]:
# --- проверка ---
assert set(cv_logloss) == set(C_grid), "ключи — все значения C_grid"
assert all(v > 0 for v in cv_logloss.values()), "log loss положителен: разверните знак neg_log_loss"
check = -cross_val_score(LogisticRegression(C=0.1, max_iter=10000), X_train, y_train, cv=cv_cancer, scoring="neg_log_loss").mean()
assert np.isclose(cv_logloss[0.1], check), "значение для C=0.1 не совпало с пересчетом"
assert best_C == min(cv_logloss, key=cv_logloss.get)
assert best_C not in (C_grid[0], C_grid[-1]), "лучший C должен быть внутри сетки, а не на краю"
print("проверки пройдены")

### Задача 18. L1 отбирает признаки

Штраф L1 обнуляет веса. В sklearn он включается параметром `l1_ratio=1.0`
и требует решателя `liblinear`. Для каждого `C` из `l1_grid` обучите на `X_train`
`LogisticRegression(l1_ratio=1.0, solver="liblinear", C=C, random_state=SEED)`
и сложите
модели в словарь `l1_models`: ключ `C`, значение — обученная модель. В словарь
`n_nonzero` запишите число ненулевых весов каждой: ключ `C`, значение — число.
В `kept_features` положите список имен признаков из `cancer_names`
с ненулевым весом при `C=0.1`, в порядке признаков.

In [ ]:
l1_grid = [0.01, 0.03, 0.1, 0.3, 1]

In [ ]:
l1_models: dict[float, LogisticRegression] = ...   # ключ — C из l1_grid
n_nonzero: dict[float, int] = ...                   # ключ — C из l1_grid
kept_features: list[str] = ...                      # имена признаков при C=0.1

print(n_nonzero)
print("при C=0.1 остались:", kept_features)

In [ ]:
# --- проверка ---
assert set(n_nonzero) == set(l1_grid)
check_l1 = LogisticRegression(l1_ratio=1.0, solver="liblinear", C=0.1, random_state=SEED).fit(X_train, y_train)
assert n_nonzero[0.1] == int(np.sum(check_l1.coef_[0] != 0)), "число ненулевых весов при C=0.1 не совпало"
assert kept_features == [n for n, w in zip(cancer_names, check_l1.coef_[0]) if w != 0], "kept_features не совпал с пересчетом"
assert n_nonzero[0.01] < n_nonzero[1], "чем слабее штраф, тем больше признаков остается"
print("проверки пройдены")

## Блок F. Три сорта

### Задача 19. Один против всех руками

Вино, все три сорта, данные готовы: `X_w_train`, `y_w_train` с метками 0, 1, 2
и тестовые `X_w_test`, `y_w_test`. Соберем схему «один против всех» руками.

1. Для каждого сорта $k \in \{0, 1, 2\}$ обучите на `X_w_train` бинарную
   `LogisticRegression(max_iter=5000)`, где целевая переменная равна единице
   для вин сорта $k$ и нулю для остальных. Три модели сложите в список
   `ovr_models` в порядке сортов.
2. У каждой модели возьмите `decision_function(X_w_test)` — это уверенность
   в том, что вино сорта $k$, по числу на тестовое вино. Сложите три вектора
   в матрицу `ovr_scores`: строка — тестовое вино, столбец $k$ — уверенность
   $k$-й модели.
3. Ответ `ovr_pred` — для каждого вина номер столбца с наибольшей уверенностью.
4. Обучите обычную многоклассовую `LogisticRegression(max_iter=5000)`
   на `X_w_train`, `y_w_train` и сохраните ее в `multinomial_model`.
5. В `agreement_share` положите долю тестовых вин, для которых ответ
   `ovr_pred` совпал с ответом `multinomial_model.predict(X_w_test)`.

In [ ]:
wine = load_wine()
X_w_train_raw, X_w_test_raw, y_w_train, y_w_test = train_test_split(
    wine.data, wine.target, test_size=0.3, random_state=SEED, stratify=wine.target
)
scaler_wine = StandardScaler().fit(X_w_train_raw)
X_w_train, X_w_test = scaler_wine.transform(X_w_train_raw), scaler_wine.transform(X_w_test_raw)

In [ ]:
ovr_models: list[LogisticRegression] = ...   # три обученные модели, в порядке сортов 0, 1, 2
ovr_scores: np.ndarray = ...                 # (n_test, 3): строка — вино, столбец k — уверенность модели k
ovr_pred: np.ndarray = ...                   # (n_test,) целые от 0 до 2

multinomial_model: LogisticRegression = ...  # обученная многоклассовая модель
agreement_share: float = ...                 # доля совпадений ovr_pred с ее ответами на тесте

print(f"accuracy один против всех {accuracy_score(y_w_test, ovr_pred):.3f}, совпадение с softmax {agreement_share:.3f}")

In [ ]:
# --- проверка ---
assert len(ovr_models) == 3 and all(hasattr(m, "coef_") for m in ovr_models), "ovr_models — три обученные модели"
assert ovr_scores.shape == (len(y_w_test), 3) and ovr_pred.shape == (len(y_w_test),)
assert hasattr(multinomial_model, "coef_") and multinomial_model.coef_.shape[0] == 3, "multinomial_model — обученная модель на трех сортах"
check_col = LogisticRegression(max_iter=5000).fit(X_w_train, (y_w_train == 2).astype(int)).decision_function(X_w_test)
assert np.allclose(ovr_scores[:, 2], check_col), "столбец 2 не совпал с классификатором «сорт 2 против остальных»"
assert np.array_equal(ovr_pred, ovr_scores.argmax(axis=1))
assert accuracy_score(y_w_test, ovr_pred) > 0.9 and 0 <= agreement_share <= 1
print("проверки пройдены")

## Итог

После лабораторной вы должны уметь объяснить:

- что такое отступ и как через него записываются функции потерь
  логистической регрессии и регрессии на метках
- почему без штрафа веса логистической регрессии на почти разделимых данных
  растут без остановки и как `C` в sklearn связан со штрафом
- когда регрессия на метках работает, а когда ломается, и почему ее выход —
  не вероятность
- почему порог выбирают по цене ошибки на отдельной выборке, а не по умолчанию
- что делает `class_weight` и чем это похоже на сдвиг порога
- как L1 отбирает признаки и чем один против всех отличается от softmax